# 10 — Tutupan lahan dan deteksi perubahan di Google Earth Engine

Tanpa mengunduh satu citra pun: komposit Landsat musim kemarau 2015 dan 2024, klasifikasi Random Forest
dengan label dari ESA WorldCover 2021, lalu luas lahan terbangun dan ekspansinya.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import geopandas as gpd
import pandas as pd

import gee
from kota import CFG, OSM, PROC

ee = gee.init()
kota = gpd.read_file(OSM, layer="batas_kota")
AOI = gee.to_ee(kota, simplify_m=30).geometry()
print("luas AOI (km²):", round(AOI.area(10).getInfo() / 1e6, 1))

## 1. Komposit Landsat musim kemarau
Landsat 8/9 Collection 2 Level-2 (reflektan permukaan), Juni–Oktober. Awan dan bayangan awan dibuang dengan
bit QA_PIXEL, lalu diambil **median** per piksel.

In [ ]:
KANAL = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]
NAMA = ["biru", "hijau", "merah", "nir", "swir1", "swir2"]


def mask_landsat(img):
    qa = img.select("QA_PIXEL")
    bersih = qa.bitwiseAnd(0b11111).eq(0)  # fill, dilated cloud, cirrus, cloud, cloud shadow
    sr = img.select(KANAL).multiply(0.0000275).add(-0.2).rename(NAMA)
    return sr.updateMask(bersih)


def komposit(tahun):
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
           .filterBounds(AOI).filterDate(f"{tahun}-06-01", f"{tahun}-10-31").map(mask_landsat))
    img = col.median().clip(AOI)
    ndvi = img.normalizedDifference(["nir", "merah"]).rename("ndvi")
    ndbi = img.normalizedDifference(["swir1", "nir"]).rename("ndbi")
    mndwi = img.normalizedDifference(["hijau", "swir1"]).rename("mndwi")
    return img.addBands([ndvi, ndbi, mndwi]), col.size()


citra = {}
for t in [2015, 2021, 2024]:
    citra[t], n = komposit(t)
    print(t, "jumlah scene:", n.getInfo())

## 2. Label latih dari ESA WorldCover 2021
WorldCover (10 m) disederhanakan menjadi empat kelas. Sampel diambil berstrata pada komposit **2021**, tahun
yang sama dengan label, lalu model diterapkan ke 2015 dan 2024.

In [ ]:
KELAS = {0: "terbangun", 1: "vegetasi", 2: "air", 3: "lahan terbuka"}
wc = ee.Image("ESA/WorldCover/v200/2021").select("Map")
label = wc.remap([50, 10, 20, 30, 40, 90, 95, 80, 60], [0, 1, 1, 1, 1, 1, 1, 2, 3]).rename("kelas")

sampel = citra[2021].addBands(label).stratifiedSample(
    numPoints=600, classBand="kelas", region=AOI, scale=30, seed=42, geometries=False)
sampel = sampel.randomColumn("acak", 42)
latih = sampel.filter(ee.Filter.lt("acak", 0.7))
uji = sampel.filter(ee.Filter.gte("acak", 0.7))
fitur = NAMA + ["ndvi", "ndbi", "mndwi"]
rf = ee.Classifier.smileRandomForest(numberOfTrees=100, seed=42).train(latih, "kelas", fitur)
print("sampel per kelas:", sampel.aggregate_histogram("kelas").getInfo())

In [ ]:
cm = uji.classify(rf).errorMatrix("kelas", "classification")
print(f"Kesesuaian dengan WorldCover (data uji 30 %): OA {cm.accuracy().getInfo():.3f}, "
      f"Kappa {cm.kappa().getInfo():.3f}")
pd.DataFrame(cm.getInfo(), index=list(KELAS.values()), columns=list(KELAS.values()))

Ini **kesesuaian dengan WorldCover**, bukan akurasi terhadap kenyataan: WorldCover sendiri punya kesalahan,
dan model diuji pada tahun yang sama dengan labelnya. Menerapkannya ke 2015 menambah ketidakpastian.

## 3. Klasifikasi 2015 dan 2024, luas per kelas

In [ ]:
kelas = {t: citra[t].classify(rf) for t in [2015, 2024]}
luas = pd.DataFrame({t: gee.luas_per_kelas(kelas[t], AOI, 30, KELAS) for t in kelas})
luas["perubahan (ha)"] = luas[2024] - luas[2015]
luas

## 4. Jebakan: perubahan dari dua klasifikasi terpisah
Luas terbangun di atas *turun* dari 2015 ke 2024 — tidak masuk akal untuk kota yang tumbuh. Hitung perubahan
kotor: piksel yang berubah menjadi terbangun, dan yang berubah dari terbangun.

In [ ]:
jadi = kelas[2015].neq(0).And(kelas[2024].eq(0))
hilang = kelas[2015].eq(0).And(kelas[2024].neq(0))
kotor = {k: v for k, v in zip(["menjadi terbangun", "terbangun menjadi lainnya"],
                              [gee.luas_per_kelas(img.selfMask(), AOI, 30, {1: "ha"})["ha"] for img in [jadi, hilang]])}
pd.Series(kotor, name="ha")

Ribuan hektar "terbangun menjadi lainnya" hampir pasti kesalahan klasifikasi, bukan pembongkaran. Dua peta
yang masing-masing 80 % benar menghasilkan peta perubahan yang jauh lebih buruk: kesalahan kedua tahun
**bertambah**, tidak saling meniadakan. Model yang dilatih pada 2021 dan diterapkan ke tahun lain menambah
kesalahan lagi (kondisi atmosfer, musim, dan sensor berbeda).

Untuk mengukur perubahan, gunakan produk yang dibuat **konsisten antar-tahun**.

## 5. GHLC: tutupan lahan harmonis 2020–2024
Global Harmonized Land Cover (OpenGeoHub; Moreno et al., 2026) menggabungkan WorldCover, GLAD GLCLUC,
dan belasan produk lain dengan aturan yang sama setiap tahun: 30 m tahunan 2020–2024, 10 m untuk 2020.
Jendela kota diunduh dengan `scripts/fetch_ghlc.py`, langsung dari COG — tanpa mengunduh berkas global.

In [ ]:
import numpy as np
import rasterio
from rasterio.features import rasterize

from kota import RAW

GRUP = {1: "terbangun", 2: "pertanian", 3: "vegetasi", 4: "air & tambak", 5: "lainnya"}


def grup(a):
    g = np.full(a.shape, 5, dtype=np.uint8)
    g[a == 1] = 1
    g[(a >= 2) & (a <= 11)] = 2
    g[(a >= 12) & (a <= 26) | (a == 37)] = 3  # pohon, semak, mangrove
    g[(a == 35) | (a == 36) | (a == 40)] = 4  # air permanen, musiman, akuakultur
    g[a == 0] = 0
    return g


kel_utm = gpd.read_file(PROC / "morfologi.gpkg", layer="kelurahan")
tahun = range(2020, 2025)
with rasterio.open(RAW / "ghlc" / "ghlc30_2020.tif") as src:
    prof, piksel_ha = src.profile, abs(src.res[0] * src.res[1]) / 1e4
    zona = rasterize(((g, i + 1) for i, g in enumerate(kel_utm.geometry)), out_shape=(src.height, src.width),
                     transform=src.transform, fill=0, dtype="int32")
ghlc = {t: rasterio.open(RAW / "ghlc" / f"ghlc30_{t}.tif").read(1) for t in tahun}
dalam = zona > 0

seri = pd.DataFrame({t: pd.Series(grup(ghlc[t])[dalam]).map(GRUP).value_counts() * piksel_ha for t in tahun})
seri.round(0)

In [ ]:
b20, b24 = ghlc[2020] == 1, ghlc[2024] == 1
pd.Series({"menjadi terbangun (ha)": (~b20 & b24 & dalam).sum() * piksel_ha,
           "terbangun menjadi lainnya (ha)": (b20 & ~b24 & dalam).sum() * piksel_ha,
           "bersih (ha)": ((b24 & dalam).sum() - (b20 & dalam).sum()) * piksel_ha}).round(0)

## 6. Ekspansi terbangun per kelurahan (GHLC 2020 → 2024)
Asal lahan dari piksel yang menjadi terbangun juga terbaca: sawah, vegetasi, atau tambak.

In [ ]:
baru = ~b20 & b24 & dalam
asal = pd.Series(grup(ghlc[2020])[baru]).map(GRUP).value_counts() * piksel_ha
print("asal lahan terbangun baru (ha):", asal.round(0).to_dict())

kel_utm["ha_terbangun_2020"] = np.bincount(zona[b20 & dalam], minlength=len(kel_utm) + 1)[1:] * piksel_ha
kel_utm["ha_terbangun_2024"] = np.bincount(zona[b24 & dalam], minlength=len(kel_utm) + 1)[1:] * piksel_ha
kel_utm["ha_baru"] = np.bincount(zona[baru], minlength=len(kel_utm) + 1)[1:] * piksel_ha
kel_utm.groupby("kecamatan")[["ha_terbangun_2020", "ha_terbangun_2024", "ha_baru"]].sum() \
    .sort_values("ha_baru", ascending=False).round(0).head(8)

In [ ]:
kel_utm[["id", "kelurahan", "kecamatan", "ha_terbangun_2020", "ha_terbangun_2024", "ha_baru", "geometry"]] \
    .to_file(PROC / "ekspansi_terbangun.gpkg", layer="kelurahan")
seri.to_csv(PROC / "ghlc_seri_2020_2024.csv")
luas.to_csv(PROC / "luas_tutupan_rf.csv")
print("disimpan: ekspansi_terbangun.gpkg, ghlc_seri_2020_2024.csv, luas_tutupan_rf.csv")